# 023-learned-correlation-head

# Imports

In [ ]:
import json
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.figure import Figure
from tqdm.auto import tqdm

from uavloc.data.drone_frames import DroneFrame, frame_place, load_frames
from uavloc.data.match_pairs import PairBankConfig, build_pair_bank, village_split
from uavloc.data.multidate import load_date
from uavloc.datasets import dataset_ref
from uavloc.features.factory import correlator_config
from uavloc.features.segmentation import SegmentationFeatures, load_segmenter
from uavloc.matching.benchmark import HARM_MARGIN_M, benchmark_draws, median_ms_per_correction, score_draws, summarise
from uavloc.matching.correlator import PositionCorrelator, weighted_coverage
from uavloc.matching.learned import LearnedScorer, TrainConfig, build_head, train_head
from uavloc.track import REPO_ROOT, experiment_dir, log_hparams, log_run, set_seed, tensorboard_writer

# Params

In [ ]:
EXP = "023-learned-correlation-head"
NOTE = ""
LOG_RUN = True

ARM = "ncc"                       # "ncc" (007.1) | "weights" (W) | "head" (H)
SEED = 42                         # head init and batch order only
DEVICE = "mps"

PLACES = ["bushcha-50p2422-25p8305", "birky-49p9105-23p9192"]
FRAMES_DATASET = "village-crossdate-frames"
HOLDOUT_DATASET = "village-crossdate-holdout"
MAP_DATE = "newest"
MATCHING = "weighted-ncc"         # ncc arm; trained arms switch to "learned"
CLASS_WEIGHTS = (0.0, 1.0, 1.0, 0.5, 0.0, 0.5)   # background, building, track, woody, field, water: ncc arm, W init, blind rule
WINDOW = "rect"
SEARCH_SCALE = 1.7
NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 10.0
N_DIRECTIONS = 8
BENCH_SEED = 42                   # draws stay identical across SEED, so every run pairs with 023.1
FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
DRONE_FEATURE_CROP_PCT = 0.0
SEG_ARCHITECTURE = "lraspp"
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
SEG_CHANNELS = "all"
MATCH_GSD_M = 0.385

EXCLUDE_PLACES = ["birky-49p9105-23p9192", "bushcha-50p2422-25p8305", "dubrova-49p5175-24p0765"]
N_VAL_VILLAGES = 10
SPLIT_SEED = 42
DRONE_DATE = "current"
ALTITUDE_M = 40.0
MARGIN_M = 80.0
N_TRAIN_PAIRS = 20_000
N_VAL_PAIRS = 2_000
BANK_SEED = 42

STEPS = 4_000
BATCH_SIZE = 32
LR = 1e-3
WARMUP_FRAC = 0.1
WEIGHT_DECAY = 1e-4
TARGET_SIGMA_CELLS = 1.0
LOG_EVERY_STEPS = 100
MONITOR_PAIRS = 512
PREVIEW_PAIRS = 8
ENCODER_CHANNELS = (32, 32, 16)
REFINE_CHANNELS = 16

TIMING_DEVICE = "cpu"
TIMING_FRAMES = 8
TIMING_REPEATS = 3

# Data

In [ ]:
set_seed(SEED)

In [ ]:
TRAINED = ARM != "ncc"
writer = tensorboard_writer(EXP) if TRAINED else None

In [ ]:
train_villages, val_villages = village_split(EXCLUDE_PLACES, N_VAL_VILLAGES, SPLIT_SEED)
len(train_villages), val_villages

In [ ]:
PAIR_CONFIG = PairBankConfig(drone_date=DRONE_DATE, altitude_m=ALTITUDE_M, fov_w_deg=FOV_W_DEG, margin_m=MARGIN_M,
                             search_scale=SEARCH_SCALE, match_gsd_m=MATCH_GSD_M,
                             sat_feature_crop_pct=SAT_FEATURE_CROP_PCT, seg_channels=SEG_CHANNELS,
                             seg_checkpoint=SEG_CHECKPOINT)
TRAIN_CONFIG = TrainConfig(steps=STEPS, batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY,
                           target_sigma_cells=TARGET_SIGMA_CELLS, log_every_steps=LOG_EVERY_STEPS,
                           monitor_pairs=MONITOR_PAIRS, preview_pairs=PREVIEW_PAIRS, seed=SEED, warmup_frac=WARMUP_FRAC)

In [ ]:
RUN_CONFIG = {
    "method": "segmentation", "arm": ARM, "matching": "learned" if TRAINED else MATCHING, "class_weights": CLASS_WEIGHTS,
    "noise_m": NOISE_M, "primary_noise_m": PRIMARY_NOISE_M, "n_directions": N_DIRECTIONS, "bench_seed": BENCH_SEED,
    "window": WINDOW, "search_scale": SEARCH_SCALE, "search_radius_m": None, "prior_sigma_m": None, "gate": "none",
    "fov_w_deg": FOV_W_DEG, "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": DRONE_FEATURE_CROP_PCT,
    "seg_architecture": SEG_ARCHITECTURE, "seg_checkpoint_sat": SEG_CHECKPOINT, "seg_checkpoint_drone": SEG_CHECKPOINT,
    "seg_channels": SEG_CHANNELS, "drone_seg_gsd_m": MATCH_GSD_M, "match_gsd_m": MATCH_GSD_M,
    "frames_dataset": FRAMES_DATASET, "frames_dataset_ref": dataset_ref(FRAMES_DATASET),
    "holdout_dataset": HOLDOUT_DATASET, "holdout_dataset_ref": dataset_ref(HOLDOUT_DATASET),
    "places": PLACES, "map_date": MAP_DATE,
    "split": {"exclude": EXCLUDE_PLACES, "n_val": N_VAL_VILLAGES, "seed": SPLIT_SEED, "val": val_villages},
    "pairs": {"n_train": N_TRAIN_PAIRS, "n_val": N_VAL_PAIRS, "seed": BANK_SEED, "drone_date": DRONE_DATE,
              "altitude_m": ALTITUDE_M, "margin_m": MARGIN_M},
    "training": {"steps": STEPS, "batch_size": BATCH_SIZE, "lr": LR, "warmup_frac": WARMUP_FRAC, "weight_decay": WEIGHT_DECAY,
                 "target_sigma_cells": TARGET_SIGMA_CELLS, "log_every_steps": LOG_EVERY_STEPS,
                 "encoder_channels": ENCODER_CHANNELS, "refine_channels": REFINE_CHANNELS} if TRAINED else None,
}

In [ ]:
def load_bench(dataset: str, places: list[str]) -> dict[str, list[DroneFrame]]:
    """Frames of `dataset` grouped by place; raises if a place has none."""
    frames = load_frames(dataset)
    bench = {place: [f for f in frames if frame_place(f) == place] for place in places}
    missing = [place for place, members in bench.items() if not members]
    if missing:
        raise ValueError(f"{dataset} has no frames for {missing}")
    return bench

In [ ]:
benches = {"": load_bench(FRAMES_DATASET, PLACES), "holdout_": load_bench(HOLDOUT_DATASET, val_villages)}
{prefix or "primary": {place: len(m) for place, m in bench.items()} for prefix, bench in benches.items()}

In [ ]:
maps = {place: load_date(place, MAP_DATE) for bench in benches.values() for place in bench}

# Model

In [ ]:
model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, DEVICE, SEG_ARCHITECTURE)

In [ ]:
banks = {split: build_pair_bank(villages, n_pairs, PAIR_CONFIG, model, DEVICE, BANK_SEED)
         for split, villages, n_pairs in [("train", train_villages, N_TRAIN_PAIRS), ("val", val_villages, N_VAL_PAIRS)]
         } if TRAINED else {}
{split: len(bank) for split, bank in banks.items()}

In [ ]:
head = build_head(ARM, CLASS_WEIGHTS, ENCODER_CHANNELS, REFINE_CHANNELS) if TRAINED else None
sum(p.numel() for p in head.parameters()) if head else 0

# Run

In [ ]:
history = train_head(head, banks["train"], banks["val"], TRAIN_CONFIG, DEVICE, writer) if TRAINED else pd.DataFrame()
history.tail(1)

In [ ]:
def make_correlator(place: str, device: str, segmenter: torch.nn.Module) -> PositionCorrelator:
    """007.1's correlator on `place`, scored by the trained head when there is one."""
    features = SegmentationFeatures(segmenter, segmenter, maps[place], SEG_CHANNELS, MATCH_GSD_M, MATCH_GSD_M, device)
    scorer = LearnedScorer(head, device) if TRAINED else None
    return PositionCorrelator(features, correlator_config(RUN_CONFIG), scorer=scorer)

In [ ]:
def bench_records(bench: dict[str, list[DroneFrame]]) -> pd.DataFrame:
    """Every frame x bearing x level of one bench, scored; same draws for every arm (BENCH_SEED)."""
    parts = []
    for place, members in tqdm(bench.items(), desc="places"):
        correlator = correlators[place]
        draws = [d for noise_m in NOISE_M for d in benchmark_draws(correlator, members, noise_m, N_DIRECTIONS, BENCH_SEED)]
        parts.append(score_draws(draws, correlator.config).assign(place=place))
    records = pd.concat(parts, ignore_index=True)
    records["harm"] = records["error_m"] > records["prior_error_m"] + HARM_MARGIN_M
    return records

In [ ]:
correlators = {place: make_correlator(place, DEVICE, model) for bench in benches.values() for place in bench}

In [ ]:
records = {prefix: bench_records(bench) for prefix, bench in benches.items()}

In [ ]:
def blind_frames(bench: dict[str, list[DroneFrame]]) -> set[str]:
    """Frames whose query holds no class weighted by the hand weights: 007.1 keeps the prior on them."""
    return {f.name for place, members in bench.items() for f in members
            if weighted_coverage(correlators[place].drone_map(f), CLASS_WEIGHTS) == 0}

In [ ]:
def run_metrics(records: pd.DataFrame, bench: dict[str, list[DroneFrame]], prefix: str) -> dict[str, float]:
    """007's metrics on all / per place / covered / blind subsets, keys prefixed per bench."""
    names = {f.name for members in bench.values() for f in members}
    blind = blind_frames(bench)
    subsets = {"all": names, "covered": names - blind, "blind": blind,
               **{place.split("-")[0]: {f.name for f in members} for place, members in bench.items()}}
    metrics = summarise(records, subsets, PRIMARY_NOISE_M)
    metrics["blind_frame_share"] = len(blind) / len(names)
    return {f"{prefix}{key}": value for key, value in metrics.items()}

In [ ]:
metrics = {key: value for prefix, bench in benches.items() for key, value in run_metrics(records[prefix], bench, prefix).items()}
{k: round(v, 3) for k, v in metrics.items() if "_at_" not in k}

In [ ]:
pd.concat({prefix or "primary": r.groupby(["place", "noise_m"])[["error_m", "harm", "reachable"]].mean()
           for prefix, r in records.items()}).round(3)

# Timing

In [ ]:
timing_model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, TIMING_DEVICE, SEG_ARCHITECTURE)
timing_correlator = make_correlator(PLACES[0], TIMING_DEVICE, timing_model)
metrics["ms_per_correction"] = median_ms_per_correction(timing_correlator, benches[""][PLACES[0]][:TIMING_FRAMES],
                                                        PRIMARY_NOISE_M, N_DIRECTIONS, TIMING_REPEATS)
metrics["head_parameters"] = sum(p.numel() for p in head.parameters()) if TRAINED else 0
metrics["ms_per_correction"]

# Plots

In [ ]:
def plot_error_cdf(records: pd.DataFrame, title: str) -> Figure:
    """Error CDF per injected level, the level marked as a dotted line."""
    fig, ax = plt.subplots(figsize=(6, 4))
    for noise_m, group in records.groupby("noise_m"):
        errors = np.sort(group["error_m"].to_numpy())
        line, = ax.plot(errors, np.linspace(0, 1, len(errors)), label=f"{noise_m:g} m injected")
        ax.axvline(noise_m, color=line.get_color(), ls=":", lw=1)
    ax.set_xlabel("position error after correction, m"); ax.set_ylabel("share of draws")
    ax.set_title(title); ax.legend(); ax.grid(alpha=.3)
    fig.tight_layout()
    return fig

In [ ]:
def plot_per_frame(records: pd.DataFrame, title: str) -> Figure:
    """Mean error per frame at the primary level, sorted."""
    per_frame = records[records["noise_m"] == PRIMARY_NOISE_M].groupby("frame")["error_m"].mean().sort_values()
    fig, ax = plt.subplots(figsize=(12, 3.5))
    per_frame.plot.bar(ax=ax, width=.8)
    ax.axhline(PRIMARY_NOISE_M, color="k", ls=":", lw=1)
    ax.set_ylabel(f"mean error @ {PRIMARY_NOISE_M:g} m, m"); ax.set_title(title)
    fig.tight_layout()
    return fig

In [ ]:
figures = [plot(r, f"{ARM}, {prefix or 'primary'} bench")
           for prefix, r in records.items() for plot in (plot_error_cdf, plot_per_frame)]

# Log

In [ ]:
def finish_run() -> str:
    """log_run closes the writer; a dry run still closes it through log_hparams."""
    if not LOG_RUN:
        log_hparams(writer, RUN_CONFIG, metrics)
        return "dry"
    return log_run(exp=EXP, config=RUN_CONFIG, metrics=metrics, note=NOTE, figures=figures,
                   seed=SEED, device=DEVICE, writer=writer)["run"]

In [ ]:
artifacts = experiment_dir(EXP) / "artifacts"
artifacts.mkdir(exist_ok=True)
run_id = finish_run()
(artifacts / "split.json").write_text(json.dumps({"train": train_villages, "val": val_villages}, indent=1))
for prefix, r in records.items():
    r.to_csv(artifacts / f"{prefix}records-{run_id}.csv", index=False)
if TRAINED:
    history.to_csv(artifacts / f"history-{run_id}.csv", index=False)
    torch.save(head.state_dict(), artifacts / f"head-{run_id}.pt")
run_id